In [ ]:
import pkg_resources
import pandas as pd
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from azure.core.exceptions import HttpResponseError
import mltable
from azure.ai.ml.entities import Data
from azure.ai.ml import automl
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.automl import ClassificationPrimaryMetrics
from azure.ai.ml import Input
from ucimlrepo import fetch_ucirepo
from mltable import DataType
import subprocess


In [ ]:
subprocess.run("azd auth login")

In [ ]:

ml_client = MLClient(
    credential = DefaultAzureCredential(),
    subscription_id = "48f83b14-32a8-4911-b629-ea5aead7fb29",
    resource_group_name =  "lm1610-rg",
    workspace_name =  "lm_portfolio"
)

In [ ]:
bank_marketing = fetch_ucirepo(id = 222)

In [ ]:
X = bank_marketing.data.features
y = bank_marketing.data.targets


In [ ]:
y.rename(columns = {
    'y' : 'Subscribed'
}, inplace = True)

In [ ]:
bank_df = X.copy()
bank_df['y'] = y.copy()

In [ ]:
bank_df

In [ ]:
bank_updated_df = bank_df[~bank_df['job'].isnull()]

In [ ]:
bank_updated_df.rename(columns = {
    "y" : 'Subscribed'
}, inplace = True)

In [ ]:
bank_updated_df.drop(['education','poutcome','contact'], axis = 1,inplace = True)

In [ ]:
bank_updated_df

In [ ]:
bank_updated_df.to_csv(
    "C:/Users/leste/OneDrive/Desktop/Microsoft VS Code/Github/MLOps-Portfolio/Data Files/bank_data_mlt/bank-marketing.csv"
)

In [ ]:
csv_path = [
    {
        "file" : "C:/Users/leste/OneDrive/Desktop/Microsoft VS Code/Github/MLOps-Portfolio/Data Files/bank_data_mlt/bank-marketing.csv"
    }
]

In [ ]:
ml_tbl = mltable.from_delimited_files(paths = csv_path)

In [ ]:
ml_tbl.show(10)

In [ ]:
# converting column types

ml_tbl.convert_column_types(
    {
        'age' : DataType.to_int(),
        'balance' : DataType.to_float()
    })

In [ ]:
df_mltable = ml_tbl.to_pandas_dataframe()

> Filtering Data

In [ ]:
df_mltable.columns

In [ ]:
# Select all columns and filter rows where age is greater than 40
df_mltable.filter(items = df_mltable.columns).filter(cond = df_mltable['age'] > 40)

In [ ]:
# Select customers whose account balance is greater than 5000 and married
df_mltable.filter(cond = (df_mltable['balance'] > 5000) & (df_mltable['marital'] == 'married'))

In [ ]:
# Finding high value customers 
df_mltable.filter(cond = (df_mltable['age'] > 30) & (df_mltable['balance'] >= 10000) & (df_mltable['housing'] == 'yes'))

In [ ]:
# Finding marketing oppurtunities
df_mltable.filter(cond = (df_mltable['loan'] == 'yes') & (df_mltable['Subscribed'] =='no'))

> Keeping & Dropping Columns

In [ ]:
# Creating a customer profile using only relevant columns
customer_profile = ml_tbl.keep_columns(['age','job','marital','education','balance'])
customer_profile.to_pandas_dataframe()


> 1. Taking random samples 

> 2. Randomly splitting the data

> 3. Take `n` rows 

> 4. Skip `n` rows

In [ ]:
sample_1 =  ml_tbl.take_random_sample(probability = 0.4 ,seed = 50)
sample_1.to_pandas_dataframe()

In [ ]:
sample_2 = ml_tbl.take_random_sample(probability = 0.3 , seed = 50)
sample_2.to_pandas_dataframe()

In [ ]:
ml_sample_test , ml_sample_train = ml_tbl.random_split(percent = 0.2 , seed = 123)
df_sample_train = ml_sample_train.to_pandas_dataframe()

In [ ]:
df_sample_test = ml_sample_test.to_pandas_dataframe()

In [ ]:
df_sample_train.isin(df_sample_test)

In [ ]:
data_overlap = pd.merge(df_sample_test,df_sample_train,how = 'inner')
data_overlap # There is no data overlap

In [ ]:
ml_tbl.take(1000).to_pandas_dataframe() #Takesthe first 1000

In [ ]:
ml_tbl.skip(1000).to_pandas_dataframe() #skips the first 1000 and takes the rest 

In [ ]:
ml_df = ml_tbl

In [ ]:
try:
    ml_tbl.save('C:/Users/leste/OneDrive/Desktop/Microsoft VS Code/Github/MLOps-Portfolio/Data Files/bank_data_mlt/MLTable')
except Exception as e:
    print('The file is already saved in the folder')

In [ ]:

ml_tbl_path = r'C:/Users/leste/OneDrive/Desktop/Microsoft VS Code/Github/MLOps-Portfolio/Data Files/bank_data_mlt'


### Defining ML-Table Name & Version

In [ ]:
mltable_name = 'Bank-MLtable'
mltable_version = '4'
mltable_job = 'UCML_Classification_v1'

In [ ]:
# ML table defination
bank_data_ml_table_def = Data(
    name = 'Bank-MLtable',
    version = '3',
    description = 'This dataset has been extracted from the UCIML repo and converted to MLtable',
    path = ml_tbl_path,
    type = AssetTypes.MLTABLE
)

In [ ]:
ml_client.data.create_or_update(data = bank_data_ml_table_def)


In [ ]:
bank_mltable_asset = ml_client.data.get(f'{mltable_name}', version = f'{mltable_version}')
bank_mltable_asset.path

In [ ]:
mltable_check = mltable.load(bank_mltable_asset.path)

In [ ]:
mltable_check.show(10)

In [ ]:
bank_ml_table = Input(
    type = AssetTypes.MLTABLE,
    path = bank_mltable_asset.id,
    mode = 'download'
)

In [ ]:
bank_ml_table

In [ ]:
classification_job = automl.classification(
    name = f'{mltable_job}',
    training_data = bank_ml_table,
    experiment_name = 'test_classification_experiment',
    target_column_name = 'Subscribed',
    primary_metric = 'accuracy', # The column to observe for performance 
    n_cross_validations = 5 ,
    validation_data_size = 0.3,
    cv_split_column_names = ['Fold_1', 'Fold_2', 'Fold_3', 'Fold_4', 'Fold_5']

)

In [ ]:
classification_job_limitations = classification_job.set_limits(
    enable_early_termination = True,
    max_trials = 6,
    timeout_minutes = 45,
    trial_timeout_minutes = 20
)

In [ ]:
return_classification_job = ml_client.jobs.create_or_update(classification_job)

In [ ]:
data = ml_client.data.list()

In [ ]:
list(ClassificationPrimaryMetrics)